# ver10 Stage 1 - 정상 vs 전체 치매: 민감도 우선 임계값 보정

이 노트북은 `ver9`의 환자 단위 5-Fold 학습 구조와 결과를 그대로 보존합니다.
마지막 섹션에서는 저장된 체크포인트를 다시 학습하지 않고 불러와,
내부 validation 데이터만으로 민감도 목표별 임계값을 선택합니다.


## 1. 환경 설정


In [1]:
import os
import re
import gc
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.executable}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"device: {device}")

Python: C:\Users\user\anaconda3\envs\alzheimer\python.exe
torch: 2.12.0.dev20260408+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
device: cuda


## 2. 실험 설정


In [2]:
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")
OUTPUT_DIR = Path(r"C:\Users\user\alzheimer\patient_level_stage1")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASSES = ["NonDemented"]
POSITIVE_CLASSES = ["VeryMildDemented", "MildDemented", "ModerateDemented"]
TASK_NAME = "non_vs_demented"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15
FOLDS_TO_RUN = [1, 2, 3, 4, 5]

TRAIN_AUG_MODE = "sepaug_4n"
ROTATION_DEGREES = 10
SHIFT_TRANSLATE = (0.05, 0.05)
ZOOM_SCALE = (0.9, 1.1)
DETERMINISTIC_AUGMENTATION = True

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True
WEIGHT_DECAY = 1e-4
BALANCE_STRATEGY = "class_weight_sqrt"

# Full fine-tuning 설정
EPOCHS = 12
FREEZE_EPOCHS = 3
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_MIN_DELTA = 1e-4

# Stable threshold grid
TARGET_VALIDATION_SENSITIVITY = 0.80
THRESHOLD_GRID = np.round(np.arange(0.20, 0.6001, 0.05), 2)

print("Stage 1: NonDemented vs Demented")
print(f"Negative classes: {NEGATIVE_CLASSES}")
print(f"Positive classes: {POSITIVE_CLASSES}")
print(f"Threshold grid: {THRESHOLD_GRID.tolist()}")

Stage 1: NonDemented vs Demented
Negative classes: ['NonDemented']
Positive classes: ['VeryMildDemented', 'MildDemented', 'ModerateDemented']
Threshold grid: [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]


## 3. 환자 단위 Manifest 생성


In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
selected_classes = NEGATIVE_CLASSES + POSITIVE_CLASSES
for class_name in selected_classes:
    target = 0 if class_name in NEGATIVE_CLASSES else 1
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), class_dir
    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, image_path.name
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert not manifest.empty
assert manifest.groupby("patient_id")["target"].nunique().max() == 1
assert manifest.groupby("patient_id")["class_name"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        class_name=("class_name", "first"),
        image_count=("image_path", "count"),
    )
)

manifest_path = OUTPUT_DIR / f"{TASK_NAME}_all_images_manifest.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
print("\n[Binary target patient counts]")
print(patient_table["target"].value_counts().sort_index())
print("\n[Original class patient counts]")
print(patient_table["class_name"].value_counts())
print(f"Manifest: {manifest_path}")

Images: 86,437
Patients: 347

[Binary target patient counts]
target
0    266
1     81
Name: count, dtype: int64

[Original class patient counts]
class_name
NonDemented         266
VeryMildDemented     58
MildDemented         21
ModerateDemented      2
Name: count, dtype: int64
Manifest: C:\Users\user\alzheimer\patient_level_stage1\non_vs_demented_all_images_manifest.csv


## 4. Dataset 및 환자 단위 평가 함수


In [4]:
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

weights = models.EfficientNet_B0_Weights.DEFAULT
preprocess = weights.transforms()

@contextmanager
def temporary_seed(seed):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    try:
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed)
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)

class PatientSliceDataset(Dataset):
    def __init__(self, dataframe, preprocess, aug_mode="original", deterministic=True, seed=42):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess
        self.deterministic = deterministic
        self.seed = seed

        if aug_mode == "original":
            self.output_types = ["original"]
        elif aug_mode == "sepaug_3n":
            self.output_types = ["rotation", "shift", "zoom"]
        elif aug_mode == "sepaug_4n":
            self.output_types = ["original", "rotation", "shift", "zoom"]
        else:
            raise ValueError(aug_mode)

        self.rotation = transforms.RandomRotation(ROTATION_DEGREES)
        self.shift = transforms.RandomAffine(degrees=0, translate=SHIFT_TRANSLATE)
        self.zoom = transforms.RandomAffine(degrees=0, scale=ZOOM_SCALE)

    def __len__(self):
        return len(self.df) * len(self.output_types)

    def _augment(self, image, aug_type):
        if aug_type == "original":
            return image
        if aug_type == "rotation":
            return self.rotation(image)
        if aug_type == "shift":
            return self.shift(image)
        if aug_type == "zoom":
            return self.zoom(image)
        raise ValueError(aug_type)

    def __getitem__(self, idx):
        multiplier = len(self.output_types)
        base_idx = idx // multiplier
        aug_idx = idx % multiplier
        row = self.df.iloc[base_idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.deterministic:
            with temporary_seed(self.seed + base_idx * 1009 + aug_idx * 9176):
                image = self._augment(image, self.output_types[aug_idx])
        else:
            image = self._augment(image, self.output_types[aug_idx])

        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )

def build_model():
    model = models.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model.to(device)

def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(prob)
        grouped_labels[patient_id] = label
    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds

def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
    }

## 5. Fine-tuning 및 기존 임계값 선택 함수


In [5]:
from sklearn.model_selection import StratifiedKFold, train_test_split

def configure_trainable_layers(model, mode):
    for parameter in model.parameters():
        parameter.requires_grad = False

    if mode == "head_only":
        pass
    elif mode == "all":
        for parameter in model.features.parameters():
            parameter.requires_grad = True
    else:
        raise ValueError(mode)

    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

def make_optimizer(model):
    head_ids = {id(p) for p in model.classifier.parameters()}
    head_params, backbone_params = [], []
    for parameter in model.parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in head_ids:
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": BACKBONE_LR})
    groups.append({"params": head_params, "lr": HEAD_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def make_class_weight(inner_train_patients):
    counts_series = inner_train_patients["target"].value_counts().sort_index()
    counts = np.array(
        [counts_series.get(0, 0), counts_series.get(1, 0)],
        dtype=np.float64,
    )
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None

def evaluate_model(model, loader):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []
    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
            probabilities = logits.softmax(dim=1)[:, 1].cpu().numpy()
            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probabilities.tolist())

    ids, y_true, y_prob, _ = aggregate_patient_predictions(
        patient_ids,
        labels_all,
        probabilities_all,
        threshold=0.5,
    )
    return ids, y_true, y_prob

def choose_stable_threshold(y_true, y_prob):
    rows = []
    for threshold in THRESHOLD_GRID:
        metrics = calculate_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append({
            "threshold": float(threshold),
            "distance_from_0_5": abs(float(threshold) - 0.5),
            **metrics,
        })

    table = pd.DataFrame(rows)
    eligible = table[
        table["sensitivity"] >= TARGET_VALIDATION_SENSITIVITY
    ].copy()
    target_reached = not eligible.empty

    if not target_reached:
        eligible = table[
            table["sensitivity"] == table["sensitivity"].max()
        ].copy()

    selected = eligible.sort_values(
        ["specificity", "precision", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(selected["threshold"]), target_reached, table

def train_one_fold(fold_number, outer_train_patients, outer_test_patients):
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])
    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    train_dataset = PatientSliceDataset(
        train_df, preprocess, TRAIN_AUG_MODE,
        DETERMINISTIC_AUGMENTATION, run_seed,
    )
    val_dataset = PatientSliceDataset(
        val_df, preprocess, "original", True, run_seed,
    )
    test_dataset = PatientSliceDataset(
        test_df, preprocess, "original", True, run_seed,
    )

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    model = build_model()
    configure_trainable_layers(model, "head_only")
    optimizer = make_optimizer(model)
    class_weight = make_class_weight(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and torch.cuda.is_available()),
    )

    best_val_auroc = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    fine_tuning_started = False

    print(
        f"\n[Stage 1 fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, "
        f"val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}"
    )

    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_EPOCHS + 1:
            configure_trainable_layers(model, "all")
            optimizer = make_optimizer(model)
            fine_tuning_started = True
            epochs_without_improvement = 0
            print("Full backbone unfreeze")

        model.train()
        loss_sum, total = 0.0, 0
        for images, labels, patient_ids in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        _, val_y_true, val_y_prob = evaluate_model(model, val_loader)
        val_metrics = calculate_metrics(val_y_true, val_y_prob, threshold=0.5)

        improved = (
            val_metrics["auroc"]
            > best_val_auroc + EARLY_STOPPING_MIN_DELTA
        )
        if improved:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        elif fine_tuning_started:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} "
            f"AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

        if (
            fine_tuning_started
            and epochs_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model = model.to(device)

    val_ids, val_y_true, val_y_prob = evaluate_model(model, val_loader)
    selected_threshold, target_reached, threshold_table = (
        choose_stable_threshold(val_y_true, val_y_prob)
    )
    val_calibrated = calculate_metrics(
        val_y_true, val_y_prob, threshold=selected_threshold
    )

    test_ids, test_y_true, test_y_prob = evaluate_model(model, test_loader)
    default_test = calculate_metrics(
        test_y_true, test_y_prob, threshold=0.5
    )
    calibrated_test = calculate_metrics(
        test_y_true, test_y_prob, threshold=selected_threshold
    )

    checkpoint_path = (
        CHECKPOINT_DIR / f"{TASK_NAME}_full_finetune_fold{fold_number}.pt"
    )
    torch.save({
        "model_state_dict": best_state,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
    }, checkpoint_path)

    result = {
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "default_sensitivity": default_test["sensitivity"],
        "default_specificity": default_test["specificity"],
        "default_f1": default_test["f1"],
        "calibrated_accuracy": calibrated_test["accuracy"],
        "calibrated_precision": calibrated_test["precision"],
        "calibrated_sensitivity": calibrated_test["sensitivity"],
        "calibrated_specificity": calibrated_test["specificity"],
        "calibrated_f1": calibrated_test["f1"],
        "calibrated_macro_f1": calibrated_test["macro_f1"],
        "auroc": calibrated_test["auroc"],
        "auprc": calibrated_test["auprc"],
    }
    print("Test:", result)

    del model, optimizer, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 6. 환자 단위 5-Fold 학습 및 결과 저장


In [6]:
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

partial_results_path = OUTPUT_DIR / f"{TASK_NAME}_5fold_partial_results.csv"
fold_assignment_path = OUTPUT_DIR / f"{TASK_NAME}_outer_5fold_assignments.csv"

if partial_results_path.exists():
    existing_results_df = pd.read_csv(partial_results_path)
    all_results = existing_results_df.to_dict("records")
    completed_folds = {int(row["fold"]) for row in all_results}
    print(f"기존 결과 {len(all_results)}개를 로드했습니다.")
    print(f"완료된 folds: {sorted(completed_folds)}")
else:
    all_results = []
    completed_folds = set()
    print("기존 결과가 없습니다. Fold 1부터 시작합니다.")

fold_assignments = []

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets),
    start=1,
):
    outer_train_patients = patient_table.iloc[
        outer_train_idx
    ].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[
        outer_test_idx
    ].reset_index(drop=True)

    for patient_id in outer_test_patients["patient_id"]:
        fold_assignments.append({
            "patient_id": patient_id,
            "outer_test_fold": fold_number,
        })

    if fold_number not in FOLDS_TO_RUN:
        continue
    if fold_number in completed_folds:
        print(f"[SKIP] 이미 완료됨: fold {fold_number}")
        continue

    print(
        f"\n===== STAGE 1 OUTER FOLD {fold_number} =====\n"
        f"train patients={len(outer_train_patients)}, "
        f"test patients={len(outer_test_patients)}\n"
        f"test target counts="
        f"{outer_test_patients['target'].value_counts().sort_index().to_dict()}"
    )

    result = train_one_fold(
        fold_number,
        outer_train_patients,
        outer_test_patients,
    )
    all_results.append(result)
    completed_folds.add(fold_number)

    temp_path = partial_results_path.with_suffix(".tmp.csv")
    pd.DataFrame(all_results).to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig",
    )
    temp_path.replace(partial_results_path)
    print(f"중간 결과 저장: {partial_results_path}")

pd.DataFrame(fold_assignments).to_csv(
    fold_assignment_path,
    index=False,
    encoding="utf-8-sig",
)

results_df = pd.DataFrame(all_results)
display(results_df.sort_values("fold"))

기존 결과 5개를 로드했습니다.
완료된 folds: [1, 2, 3, 4, 5]
[SKIP] 이미 완료됨: fold 1
[SKIP] 이미 완료됨: fold 2
[SKIP] 이미 완료됨: fold 3
[SKIP] 이미 완료됨: fold 4
[SKIP] 이미 완료됨: fold 5


,fold,best_epoch,val_auroc,selected_threshold,validation_target_reached,test_patients,default_sensitivity,default_specificity,default_f1,calibrated_accuracy,calibrated_precision,calibrated_sensitivity,calibrated_specificity,calibrated_f1,calibrated_macro_f1,auroc,auprc
0,1,11,0.950000,0.55,True,70,0.812500,0.833333,0.684211,0.857143,0.666667,0.750000,0.888889,0.705882,0.805771,0.920139,0.747139
1,2,6,0.909375,0.50,False,70,0.705882,0.962264,0.774194,0.900000,0.857143,0.705882,0.962264,0.774194,0.854987,0.970033,0.901397
2,3,4,0.915625,0.60,True,69,0.687500,0.811321,0.594595,0.840580,0.666667,0.625000,0.905660,0.645161,0.771179,0.891509,0.681448
3,4,1,0.965625,0.45,True,69,0.625000,0.849057,0.588235,0.797101,0.545455,0.750000,0.811321,0.631579,0.745789,0.880896,0.657154
4,5,4,0.978125,0.45,True,69,0.687500,0.905660,0.687500,0.840580,0.647059,0.687500,0.886792,0.666667,0.780952,0.898585,0.681042


## 7. 기존 ver9 결과 요약


In [7]:
assert not results_df.empty

metric_columns = [
    "default_sensitivity",
    "default_specificity",
    "default_f1",
    "calibrated_accuracy",
    "calibrated_precision",
    "calibrated_sensitivity",
    "calibrated_specificity",
    "calibrated_f1",
    "calibrated_macro_f1",
    "auroc",
    "auprc",
    "selected_threshold",
]

summary_rows = []
for metric in metric_columns:
    summary_rows.append({
        "metric": metric,
        "mean": results_df[metric].mean(),
        "std": results_df[metric].std(ddof=1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"{TASK_NAME}_5fold_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_df)
print(
    f"Default sensitivity: "
    f"{results_df['default_sensitivity'].mean():.4f}"
)
print(
    f"Calibrated sensitivity: "
    f"{results_df['calibrated_sensitivity'].mean():.4f}"
)
print(
    f"Calibrated specificity: "
    f"{results_df['calibrated_specificity'].mean():.4f}"
)
print(
    f"AUROC: {results_df['auroc'].mean():.4f} ± "
    f"{results_df['auroc'].std(ddof=1):.4f}"
)
print(
    f"AUPRC: {results_df['auprc'].mean():.4f} ± "
    f"{results_df['auprc'].std(ddof=1):.4f}"
)
print(f"Summary saved: {summary_path}")

,metric,mean,std
0,default_sensitivity,0.703676,0.068119
1,default_specificity,0.872327,0.061204
2,default_f1,0.665747,0.076885
3,calibrated_accuracy,0.847081,0.037028
4,calibrated_precision,0.676598,0.112766
5,calibrated_sensitivity,0.703676,0.051837
6,calibrated_specificity,0.890985,0.054001
7,calibrated_f1,0.684697,0.057376
8,calibrated_macro_f1,0.791736,0.041375
9,auroc,0.912233,0.035359


Default sensitivity: 0.7037
Calibrated sensitivity: 0.7037
Calibrated specificity: 0.8910
AUROC: 0.9122 ± 0.0354
AUPRC: 0.7336 ± 0.0996
Summary saved: C:\Users\user\alzheimer\patient_level_stage1\non_vs_demented_5fold_summary.csv


## 8. ver9 결과 해석

- AUROC는 정상과 전체 치매 환자의 순위를 구분하는 능력을 나타냅니다.
- Stage 1은 선별 단계이므로 정확도보다 **민감도**를 우선해서 확인합니다.
- 아래 ver10 섹션은 outer test를 임계값 선택에 사용하지 않습니다.


## 9. ver10 민감도 우선 임계값 보정 설정

기존 체크포인트를 재사용하므로 CNN을 다시 학습하지 않습니다.

- 각 Fold의 **inner validation**에서만 임계값을 선택합니다.
- outer test는 선택된 임계값의 일반화 성능을 평가하는 데만 사용합니다.
- 민감도 목표를 높이면 놓치는 치매 환자는 줄지만 정상 환자의 오탐은 늘 수 있습니다.
- `MIN_VALIDATION_SPECIFICITY`는 민감도를 높이는 과정에서 특이도가 지나치게 무너지는 것을 막는 안전장치입니다.


In [8]:
# 민감도 우선 보정 설정
SENSITIVITY_TARGETS = [0.80, 0.85, 0.90]
MIN_VALIDATION_SPECIFICITY = 0.65

# 기존 0.05 간격보다 세밀하게 탐색합니다.
SENSITIVITY_THRESHOLD_GRID = np.round(
    np.arange(0.10, 0.7001, 0.025),
    3,
)

SENSITIVITY_OUTPUT_DIR = OUTPUT_DIR / "sensitivity_first"
SENSITIVITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"민감도 목표: {SENSITIVITY_TARGETS}")
print(
    "임계값 탐색 범위:",
    float(SENSITIVITY_THRESHOLD_GRID.min()),
    "~",
    float(SENSITIVITY_THRESHOLD_GRID.max()),
)
print(f"최소 validation 특이도: {MIN_VALIDATION_SPECIFICITY:.2f}")
print(f"결과 저장 폴더: {SENSITIVITY_OUTPUT_DIR}")


민감도 목표: [0.8, 0.85, 0.9]
임계값 탐색 범위: 0.1 ~ 0.7
최소 validation 특이도: 0.65
결과 저장 폴더: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first


## 10. 민감도 우선 임계값 선택 및 체크포인트 평가 함수

임계값은 validation 민감도 목표를 만족하는 후보 중에서 선택합니다.
목표를 만족하는 후보가 여러 개라면 특이도가 가장 높은 값을 사용합니다.

validation에서도 목표를 달성하지 못하면 민감도가 가장 높은 후보를 사용하되,
`validation_target_reached=False`로 기록하여 결과를 과대 해석하지 않도록 합니다.


In [9]:
def load_trusted_checkpoint(checkpoint_path):
    # 이 노트북이 직접 생성한 로컬 체크포인트만 불러옵니다.
    # PyTorch 2.6부터 torch.load의 기본 weights_only=True 때문에
    # numpy 값이 포함된 기존 체크포인트에서 UnpicklingError가 날 수 있습니다.
    try:
        return torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # weights_only 인자가 없는 이전 PyTorch 버전 호환
        return torch.load(checkpoint_path, map_location="cpu")


def build_threshold_table(y_true, y_prob, threshold_grid):
    rows = []
    for threshold in threshold_grid:
        metrics = calculate_metrics(
            y_true,
            y_prob,
            threshold=float(threshold),
        )
        rows.append({
            "threshold": float(threshold),
            "distance_from_0_5": abs(float(threshold) - 0.5),
            **metrics,
        })
    return pd.DataFrame(rows)


def choose_sensitivity_first_threshold(
    threshold_table,
    target_sensitivity,
    min_specificity=0.65,
):
    eligible = threshold_table[
        (threshold_table["sensitivity"] >= target_sensitivity)
        & (threshold_table["specificity"] >= min_specificity)
    ].copy()
    target_reached = not eligible.empty

    if not target_reached:
        # 특이도 안전장치를 만족하는 범위에서 가능한 최대 민감도를 선택합니다.
        eligible = threshold_table[
            threshold_table["specificity"] >= min_specificity
        ].copy()
        if eligible.empty:
            eligible = threshold_table.copy()
        eligible = eligible[
            eligible["sensitivity"] == eligible["sensitivity"].max()
        ].copy()

    selected = eligible.sort_values(
        ["specificity", "precision", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(selected["threshold"]), bool(target_reached)


def make_evaluation_loaders(
    outer_train_patients,
    outer_test_patients,
    fold_number,
):
    run_seed = SEED + fold_number * 100
    _, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    val_ids = set(inner_val_patients["patient_id"])
    test_ids = set(outer_test_patients["patient_id"])
    assert val_ids.isdisjoint(test_ids)

    val_df = manifest[manifest["patient_id"].isin(val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(test_ids)].copy()

    val_dataset = PatientSliceDataset(
        val_df, preprocess, "original", True, run_seed,
    )
    test_dataset = PatientSliceDataset(
        test_df, preprocess, "original", True, run_seed,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    return val_loader, test_loader


## 11. 저장된 5-Fold 체크포인트로 민감도 목표 비교

이 셀은 학습을 수행하지 않고 추론만 수행합니다.
Fold별 validation에서 선택한 임계값을 해당 Fold의 outer test에 적용합니다.


In [10]:
sensitivity_results = []
threshold_detail_frames = []
patient_prediction_frames = []

outer_cv_for_calibration = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv_for_calibration.split(patient_indices, patient_targets),
    start=1,
):
    print(f"\n===== 민감도 보정 FOLD {fold_number} =====")

    outer_train_patients = patient_table.iloc[
        outer_train_idx
    ].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[
        outer_test_idx
    ].reset_index(drop=True)

    val_loader, test_loader = make_evaluation_loaders(
        outer_train_patients,
        outer_test_patients,
        fold_number,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{TASK_NAME}_full_finetune_fold{fold_number}.pt"
    )
    assert checkpoint_path.exists(), f"체크포인트 없음: {checkpoint_path}"

    checkpoint = load_trusted_checkpoint(checkpoint_path)
    model = build_model()
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    val_ids, val_y_true, val_y_prob = evaluate_model(model, val_loader)
    test_ids, test_y_true, test_y_prob = evaluate_model(model, test_loader)

    val_threshold_table = build_threshold_table(
        val_y_true,
        val_y_prob,
        SENSITIVITY_THRESHOLD_GRID,
    )
    val_threshold_table.insert(0, "fold", fold_number)
    threshold_detail_frames.append(val_threshold_table)

    patient_prediction_frames.append(pd.DataFrame({
        "fold": fold_number,
        "patient_id": test_ids,
        "target": test_y_true,
        "probability": test_y_prob,
    }))

    for target_sensitivity in SENSITIVITY_TARGETS:
        selected_threshold, target_reached = (
            choose_sensitivity_first_threshold(
                val_threshold_table,
                target_sensitivity,
                MIN_VALIDATION_SPECIFICITY,
            )
        )
        val_metrics = calculate_metrics(
            val_y_true,
            val_y_prob,
            selected_threshold,
        )
        test_metrics = calculate_metrics(
            test_y_true,
            test_y_prob,
            selected_threshold,
        )

        sensitivity_results.append({
            "fold": fold_number,
            "target_sensitivity": target_sensitivity,
            "selected_threshold": selected_threshold,
            "validation_target_reached": target_reached,
            "val_sensitivity": val_metrics["sensitivity"],
            "val_specificity": val_metrics["specificity"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_sensitivity": test_metrics["sensitivity"],
            "test_specificity": test_metrics["specificity"],
            "test_f1": test_metrics["f1"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_auroc": test_metrics["auroc"],
            "test_auprc": test_metrics["auprc"],
        })

        print(
            f"목표 {target_sensitivity:.2f} | "
            f"threshold {selected_threshold:.3f} | "
            f"val sens/spec "
            f"{val_metrics['sensitivity']:.3f}/"
            f"{val_metrics['specificity']:.3f} | "
            f"test sens/spec "
            f"{test_metrics['sensitivity']:.3f}/"
            f"{test_metrics['specificity']:.3f}"
        )

    del model, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sensitivity_results_df = pd.DataFrame(sensitivity_results)
threshold_details_df = pd.concat(
    threshold_detail_frames,
    ignore_index=True,
)
oof_predictions_df = pd.concat(
    patient_prediction_frames,
    ignore_index=True,
)

sensitivity_results_path = (
    SENSITIVITY_OUTPUT_DIR / "sensitivity_target_fold_results.csv"
)
threshold_details_path = (
    SENSITIVITY_OUTPUT_DIR / "validation_threshold_details.csv"
)
oof_predictions_path = (
    SENSITIVITY_OUTPUT_DIR / "outer_fold_patient_predictions.csv"
)

sensitivity_results_df.to_csv(
    sensitivity_results_path,
    index=False,
    encoding="utf-8-sig",
)
threshold_details_df.to_csv(
    threshold_details_path,
    index=False,
    encoding="utf-8-sig",
)
oof_predictions_df.to_csv(
    oof_predictions_path,
    index=False,
    encoding="utf-8-sig",
)

display(sensitivity_results_df)
print(f"Fold 결과 저장: {sensitivity_results_path}")
print(f"Validation grid 저장: {threshold_details_path}")
print(f"환자별 OOF 예측 저장: {oof_predictions_path}")



===== 민감도 보정 FOLD 1 =====
목표 0.80 | threshold 0.525 | val sens/spec 0.800/0.969 | test sens/spec 0.750/0.889
목표 0.85 | threshold 0.500 | val sens/spec 0.900/0.938 | test sens/spec 0.812/0.833
목표 0.90 | threshold 0.500 | val sens/spec 0.900/0.938 | test sens/spec 0.812/0.833

===== 민감도 보정 FOLD 2 =====
목표 0.80 | threshold 0.125 | val sens/spec 0.800/0.875 | test sens/spec 0.941/0.906
목표 0.85 | threshold 0.125 | val sens/spec 0.800/0.875 | test sens/spec 0.941/0.906
목표 0.90 | threshold 0.125 | val sens/spec 0.800/0.875 | test sens/spec 0.941/0.906

===== 민감도 보정 FOLD 3 =====
목표 0.80 | threshold 0.700 | val sens/spec 0.800/0.906 | test sens/spec 0.625/0.925
목표 0.85 | threshold 0.600 | val sens/spec 0.900/0.875 | test sens/spec 0.625/0.906
목표 0.90 | threshold 0.600 | val sens/spec 0.900/0.875 | test sens/spec 0.625/0.906

===== 민감도 보정 FOLD 4 =====
목표 0.80 | threshold 0.475 | val sens/spec 0.800/0.938 | test sens/spec 0.688/0.830
목표 0.85 | threshold 0.450 | val sens/spec 1.000/0.906 | test s

,fold,target_sensitivity,selected_threshold,validation_target_reached,val_sensitivity,val_specificity,test_accuracy,test_precision,test_sensitivity,test_specificity,test_f1,test_macro_f1,test_auroc,test_auprc
0,1,0.80,0.525,True,0.8,0.96875,0.857143,0.666667,0.750000,0.888889,0.705882,0.805771,0.920139,0.747139
1,1,0.85,0.500,True,0.9,0.93750,0.828571,0.590909,0.812500,0.833333,0.684211,0.783282,0.920139,0.747139
2,1,0.90,0.500,True,0.9,0.93750,0.828571,0.590909,0.812500,0.833333,0.684211,0.783282,0.920139,0.747139
3,2,0.80,0.125,True,0.8,0.87500,0.914286,0.761905,0.941176,0.905660,0.842105,0.891641,0.970033,0.901397
4,2,0.85,0.125,False,0.8,0.87500,0.914286,0.761905,0.941176,0.905660,0.842105,0.891641,0.970033,0.901397
5,2,0.90,0.125,False,0.8,0.87500,0.914286,0.761905,0.941176,0.905660,0.842105,0.891641,0.970033,0.901397
6,3,0.80,0.700,True,0.8,0.90625,0.855072,0.714286,0.625000,0.924528,0.666667,0.787037,0.891509,0.681448
7,3,0.85,0.600,True,0.9,0.87500,0.840580,0.666667,0.625000,0.905660,0.645161,0.771179,0.891509,0.681448
8,3,0.90,0.600,True,0.9,0.87500,0.840580,0.666667,0.625000,0.905660,0.645161,0.771179,0.891509,0.681448
9,4,0.80,0.475,True,0.8,0.93750,0.797101,0.550000,0.687500,0.830189,0.611111,0.736928,0.880896,0.657154


Fold 결과 저장: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\sensitivity_target_fold_results.csv
Validation grid 저장: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\validation_threshold_details.csv
환자별 OOF 예측 저장: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\outer_fold_patient_predictions.csv


## 12. 민감도 목표별 5-Fold 평균 비교

Stage 1 선별 모델에서는 다음 순서로 확인합니다.

1. 평균 test 민감도가 충분히 올라갔는지 확인합니다.
2. 그때 특이도가 임상적으로 감당 가능한 수준인지 확인합니다.
3. Fold별 민감도 편차가 지나치게 크지 않은지 확인합니다.
4. 목표를 정한 뒤 Fold별 선택 임계값의 중앙값을 최종 후보로 기록합니다.

outer test 결과를 보고 같은 데이터에 맞춰 임계값을 계속 조정하면 낙관적인 결과가 될 수 있습니다.
따라서 이 표는 정책 후보를 비교하는 용도로 사용하고, 최종 임계값은 별도 hold-out 데이터에서 다시 검증해야 합니다.


In [11]:
summary_rows = []
for target_sensitivity, group in sensitivity_results_df.groupby(
    "target_sensitivity"
):
    summary_rows.append({
        "target_sensitivity": target_sensitivity,
        "target_reached_folds": int(
            group["validation_target_reached"].sum()
        ),
        "median_threshold": group["selected_threshold"].median(),
        "mean_threshold": group["selected_threshold"].mean(),
        "mean_test_sensitivity": group["test_sensitivity"].mean(),
        "std_test_sensitivity": group["test_sensitivity"].std(ddof=1),
        "mean_test_specificity": group["test_specificity"].mean(),
        "std_test_specificity": group["test_specificity"].std(ddof=1),
        "mean_test_precision": group["test_precision"].mean(),
        "mean_test_f1": group["test_f1"].mean(),
        "mean_test_macro_f1": group["test_macro_f1"].mean(),
        "mean_test_auroc": group["test_auroc"].mean(),
        "mean_test_auprc": group["test_auprc"].mean(),
    })

sensitivity_summary_df = pd.DataFrame(summary_rows).sort_values(
    "target_sensitivity"
)
sensitivity_summary_path = (
    SENSITIVITY_OUTPUT_DIR / "sensitivity_target_summary.csv"
)
sensitivity_summary_df.to_csv(
    sensitivity_summary_path,
    index=False,
    encoding="utf-8-sig",
)

display(sensitivity_summary_df)

print("\n[해석 기준]")
print("- 민감도 0.80 이상을 우선 확인합니다.")
print("- 특이도가 0.70 아래로 크게 내려가면 오탐 부담을 함께 검토합니다.")
print("- median_threshold는 다음 독립 검증용 임계값 후보입니다.")
print(f"요약 저장: {sensitivity_summary_path}")


,target_sensitivity,target_reached_folds,median_threshold,mean_threshold,mean_test_sensitivity,std_test_sensitivity,mean_test_specificity,std_test_specificity,mean_test_precision,mean_test_f1,mean_test_macro_f1,mean_test_auroc,mean_test_auprc
0,0.80,5,0.475,0.455,0.738235,0.121752,0.887212,0.035311,0.667983,0.698486,0.800466,0.912233,0.733636
1,0.85,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636
2,0.90,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636



[해석 기준]
- 민감도 0.80 이상을 우선 확인합니다.
- 특이도가 0.70 아래로 크게 내려가면 오탐 부담을 함께 검토합니다.
- median_threshold는 다음 독립 검증용 임계값 후보입니다.
요약 저장: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\sensitivity_target_summary.csv


## 13. 고정 임계값 진단표

아래 표는 임계값이 변할 때 민감도와 특이도가 어떻게 교환되는지 확인하기 위한 진단용입니다.
이 outer-fold 결과에서 가장 좋아 보이는 값을 다시 같은 데이터의 최종 임계값으로 확정하면 데이터 누수가 될 수 있습니다.


In [12]:
diagnostic_rows = []
for threshold in np.round(np.arange(0.25, 0.5501, 0.025), 3):
    fold_metrics = []
    for fold_number, fold_df in oof_predictions_df.groupby("fold"):
        metrics = calculate_metrics(
            fold_df["target"].to_numpy(),
            fold_df["probability"].to_numpy(),
            threshold=float(threshold),
        )
        fold_metrics.append(metrics)

    diagnostic_rows.append({
        "threshold": float(threshold),
        "mean_sensitivity": np.mean(
            [row["sensitivity"] for row in fold_metrics]
        ),
        "std_sensitivity": np.std(
            [row["sensitivity"] for row in fold_metrics],
            ddof=1,
        ),
        "mean_specificity": np.mean(
            [row["specificity"] for row in fold_metrics]
        ),
        "mean_precision": np.mean(
            [row["precision"] for row in fold_metrics]
        ),
        "mean_f1": np.mean(
            [row["f1"] for row in fold_metrics]
        ),
    })

fixed_threshold_diagnostic_df = pd.DataFrame(diagnostic_rows)
display(fixed_threshold_diagnostic_df)

diagnostic_path = (
    SENSITIVITY_OUTPUT_DIR / "fixed_threshold_diagnostic.csv"
)
fixed_threshold_diagnostic_df.to_csv(
    diagnostic_path,
    index=False,
    encoding="utf-8-sig",
)
print(f"진단표 저장: {diagnostic_path}")


,threshold,mean_sensitivity,std_sensitivity,mean_specificity,mean_precision,mean_f1
0,0.250,0.864706,0.079933,0.759399,0.544269,0.657892
1,0.275,0.852206,0.069969,0.770720,0.550168,0.659954
2,0.300,0.852206,0.069969,0.785814,0.569532,0.672948
3,0.325,0.828676,0.096699,0.793291,0.572518,0.664302
4,0.350,0.816176,0.093190,0.808386,0.584231,0.669812
5,0.375,0.778676,0.090817,0.819706,0.585323,0.659217
6,0.400,0.766176,0.077581,0.830957,0.603081,0.664407
7,0.425,0.766176,0.077581,0.842208,0.618397,0.674802
8,0.450,0.741176,0.079048,0.849755,0.622250,0.667308
9,0.475,0.728676,0.082184,0.857303,0.631247,0.667381


진단표 저장: C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\fixed_threshold_diagnostic.csv
